# Configs

In [1]:
import pandas as pd
import numpy as np

In [2]:
Proj_path = '/Users/cmartin/Fantasy_Baseball/Ottoneu_Baseball_Projects/FG_2026_Projections'

In [3]:
Proj_date = 'Jan23_2026'

In [4]:
Projections_dates = {
    'ATC':'Jan23_2026',
    'OOPSY':'Jan23_2026'
}

In [5]:
Proj_weights = {
    'ATC':.4,
    'OOPSY':.6
}

In [6]:
Hitting_Possitions = [
    'C',
    '1B',
    '2B',
    'SS',
    '3B',
    'OF',
    'DH'
]
Pitching_Possitions = [
    'SP',
    'RP'
]

In [7]:
Player_id_cols = [
    'PlayerId','MLBAMID','Name','Team','NameASCII'
]
Hitter_Count_Stats = [
    'G','PA','AB','H','1B','2B','3B','HR','R','RBI','BB','HBP','SF','WAR','ADP'
]
Pitcher_Count_Stats = [
    'W', 'L', 'QS', 'G', 'GS', 'SV', 'HLD', 'IP', 'TBF', 'H', 'R', 'ER', 'HR', 'BB', 'HBP', 'SO','WAR','ADP'
] # 'BS',

In [8]:
def w_avg(df, values, weights):
    d = df[values]
    w = df[weights]
    return (d * w).sum() / w.sum()

In [9]:
Hitter_Projections_df = pd.DataFrame()
for proj,date in Projections_dates.items():
    this_proj_df = pd.DataFrame()
    for pos in Hitting_Possitions:
        temp_df = pd.read_csv(f"{Proj_path}/{date}/{proj}/fangraphs-{proj}_{date}-projections_{pos}.csv")
        temp_df['Projection'] = proj
        temp_df['Proj_weight'] = Proj_weights[proj]
        temp_df['POS'] = pos
        this_proj_df = pd.concat([this_proj_df, temp_df], ignore_index=True)
    this_proj_df = this_proj_df[Player_id_cols+Hitter_Count_Stats+['Projection','Proj_weight']].drop_duplicates().merge(
                this_proj_df.groupby(Player_id_cols+Hitter_Count_Stats)['POS'].agg(list).reset_index()
        )
    Hitter_Projections_df = pd.concat([Hitter_Projections_df,this_proj_df], ignore_index=True)
Hitter_Projections_df = Hitter_Projections_df[Player_id_cols+Hitter_Count_Stats+['Projection','Proj_weight']].drop_duplicates().merge(
                Hitter_Projections_df.groupby(Player_id_cols+Hitter_Count_Stats+['Projection','Proj_weight'])['POS'].agg(list).reset_index()
        )
Pitcher_Projections_df = pd.DataFrame()
for proj,date in Projections_dates.items():
    this_proj_df = pd.DataFrame()
    for pos in Pitching_Possitions:
        temp_df = pd.read_csv(f"{Proj_path}/{date}/{proj}/fangraphs-{proj}_{date}-projections_{pos}.csv")
        temp_df['Projection'] = proj
        temp_df['Proj_weight'] = Proj_weights[proj]
        temp_df['POS'] = pos
        this_proj_df = pd.concat([this_proj_df, temp_df], ignore_index=True)
    this_proj_df = this_proj_df[Player_id_cols+Pitcher_Count_Stats+['Projection','Proj_weight']].drop_duplicates().merge(
                this_proj_df.groupby(Player_id_cols+Pitcher_Count_Stats)['POS'].agg(list).reset_index()
        )
    Pitcher_Projections_df = pd.concat([Pitcher_Projections_df,this_proj_df], ignore_index=True)
Pitcher_Projections_df = Pitcher_Projections_df[Player_id_cols+Pitcher_Count_Stats+['Projection','Proj_weight']].drop_duplicates().merge(
                Pitcher_Projections_df.groupby(Player_id_cols+Pitcher_Count_Stats+['Projection','Proj_weight'])['POS'].agg(list).reset_index()
        )

In [10]:
for col in Player_id_cols:
    Hitter_Projections_df[col] = Hitter_Projections_df[col].astype(str)
    Pitcher_Projections_df[col] = Pitcher_Projections_df[col].astype(str)

In [11]:
Pitcher_Projections_df.head()

,PlayerId,MLBAMID,Name,Team,NameASCII,W,L,QS,G,GS,...,ER,HR,BB,HBP,SO,WAR,ADP,Projection,Proj_weight,POS
0,22267,669373,Tarik Skubal,DET,Tarik Skubal,14.3742,7.04161,18.1585,29.8766,29.8766,...,58.4961,18.7818,38.9320,6.39001,229.190,5.84816,5.840000,ATC,0.4,[[SP]]
1,33677,694973,Paul Skenes,PIT,Paul Skenes,12.3671,8.20263,17.6124,30.2106,30.2106,...,57.2135,15.5018,45.5249,6.72205,224.303,5.78243,9.610000,ATC,0.4,[[SP]]
2,27463,676979,Garrett Crochet,BOS,Garrett Crochet,13.2469,8.14561,15.5400,29.3089,29.3089,...,61.6010,19.3156,46.3108,5.27536,221.171,5.01687,10.100000,ATC,0.4,[[SP]]
3,17995,657277,Logan Webb,SFG,Logan Webb,12.9634,10.14590,17.1698,30.3029,30.3029,...,72.5313,15.7501,44.8790,5.85560,183.708,4.80906,56.419998,ATC,0.4,[[SP]]
4,20778,650911,Cristopher Sánchez,PHI,Cristopher Sanchez,12.3766,7.77394,16.5268,29.3337,29.3337,...,65.7470,16.0440,46.2015,5.82465,179.174,4.46801,27.580000,ATC,0.4,[[SP]]


In [12]:
Hitter_weighted_averages = Hitter_Projections_df.groupby(Player_id_cols).apply(lambda x: pd.Series(np.average(x[Hitter_Count_Stats], weights=x["Proj_weight"], axis=0), Hitter_Count_Stats)).reset_index()
Pitcher_weighted_averages = Pitcher_Projections_df.groupby(Player_id_cols).apply(lambda x: pd.Series(np.average(x[Pitcher_Count_Stats], weights=x["Proj_weight"], axis=0), Pitcher_Count_Stats)).reset_index()

In [13]:
Hitter_weighted_averages.shape[0]

1981

In [14]:
Pitcher_weighted_averages.shape[0]

3041

In [15]:
Hitter_Projections_df['POS'] = Hitter_Projections_df['POS'].apply(lambda pos_list_of_lists: pos_list_of_lists[0])
Pitcher_Projections_df['POS'] = Pitcher_Projections_df['POS'].apply(lambda pos_list_of_lists: pos_list_of_lists[0])

In [16]:
Hitter_pos_merged = Hitter_Projections_df.groupby(Player_id_cols)['POS'].apply(lambda pos_list: list(set(sum(pos_list,[])))).reset_index()
Pitcher_pos_merged = Pitcher_Projections_df.groupby(Player_id_cols)['POS'].apply(lambda pos_list: list(set(sum(pos_list,[])))).reset_index()

In [17]:
Hitter_pos_merged.shape[0]

1981

In [18]:
Pitcher_pos_merged.shape[0]

3041

In [19]:
Hitter_final_merge_df=Hitter_weighted_averages.merge(Hitter_pos_merged,on=Player_id_cols,how='outer')
Pitcher_final_merge_df=Pitcher_weighted_averages.merge(Pitcher_pos_merged,on=Player_id_cols,how='outer')

In [20]:
# Find Duplicate Players
Hitter_final_merge_df[Hitter_final_merge_df['PlayerId'].isin([key for key, val in Hitter_final_merge_df['PlayerId'].value_counts().to_dict().items() if val > 1])].sort_values('Name')

,PlayerId,MLBAMID,Name,Team,NameASCII,G,PA,AB,H,1B,...,3B,HR,R,RBI,BB,HBP,SF,WAR,ADP,POS


In [21]:
Pitcher_final_merge_df[Pitcher_final_merge_df['PlayerId'].isin([key for key, val in Pitcher_final_merge_df['PlayerId'].value_counts().to_dict().items() if val > 1])].sort_values('Name')

,PlayerId,MLBAMID,Name,Team,NameASCII,W,L,QS,G,GS,...,H,R,ER,HR,BB,HBP,SO,WAR,ADP,POS


In [22]:
# Fix known Pitcher Hitter overlaps, manually choosing which position for these players.
# Note: This explicitly leaves some players as duplicates because they are 2-way players. (e.g. Otahni)

PlayerIds_drop_Pitching = []
PlayerIds_drop_Hitting = []

# Andrés Nolaya sa3018563
PlayerIds_drop_Pitching.append('sa3018563')

# Ben Malgeri sa3016975
PlayerIds_drop_Pitching.append('sa3016975')

# Caleb Pendleton sa3022483
PlayerIds_drop_Pitching.append('sa3022483')

# Carlos Franco sa3015580
PlayerIds_drop_Pitching.append('sa3015580')

# Chase Valentine sa3020002
PlayerIds_drop_Pitching.append('sa3020002')

# Curtis Washington Jr. sa3019803	
PlayerIds_drop_Pitching.append('sa3019803')

# David Leon sa3016336
PlayerIds_drop_Pitching.append('sa3016336')

# Diego Viloria sa3016299
PlayerIds_drop_Hitting.append('sa3016299')

# Fernando Caldera sa3015547
PlayerIds_drop_Pitching.append('sa3015547')

# Jackson Cluff sa3010342
PlayerIds_drop_Hitting.append('sa3010342')

# Jesus Castro sa3019090
PlayerIds_drop_Hitting.append('sa3019090')

# Johan Contreras sa3019217
PlayerIds_drop_Pitching.append('sa3019217')

# Jose Gerardo sa3018645
PlayerIds_drop_Hitting.append('sa3018645')

# Keduar Trujillo sa3019764
PlayerIds_drop_Pitching.append('sa3019764')

# Sean Barnett sa3025496
PlayerIds_drop_Hitting.append('sa3025496')

# Wyatt Hoffman sa3018219
PlayerIds_drop_Pitching.append('sa3018219')

# Yolmer Sánchez 11602
PlayerIds_drop_Pitching.append('11602')

In [23]:
Pitcher_final_merge_df = Pitcher_final_merge_df[~Pitcher_final_merge_df['PlayerId'].isin(PlayerIds_drop_Pitching)]
Hitter_final_merge_df = Hitter_final_merge_df[~Hitter_final_merge_df['PlayerId'].isin(PlayerIds_drop_Hitting)]

In [24]:
#Test Hitter / Pitcher Concat
Quick_test = pd.concat([Hitter_final_merge_df,Pitcher_final_merge_df])
Quick_test[Quick_test['PlayerId'].isin([key for key, val in Quick_test['PlayerId'].value_counts().to_dict().items() if val > 1])].sort_values('Name')

,PlayerId,MLBAMID,Name,Team,NameASCII,G,PA,AB,H,1B,...,W,L,QS,GS,SV,HLD,IP,TBF,ER,SO
169,19755,660271,Shohei Ohtani,LAD,Shohei Ohtani,154.5648,675.76,576.206,164.76240,82.63564,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
239,19755,660271,Shohei Ohtani,LAD,Shohei Ohtani,22.1428,NaN,NaN,87.13748,NaN,...,8.979808,5.4617,11.495048,22.1428,0.0,0.0,111.9524,457.8004,39.4536,135.4648


In [25]:
Hitter_final_merge_df.to_csv(Proj_path+f'/my_Hitter_Proj_{Proj_date}.csv',index=False)

In [26]:
Pitcher_final_merge_df.to_csv(Proj_path+f'/my_Pitcher_Proj_{Proj_date}.csv',index=False)